<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# E4's Pilot: Hebb's Rule, Its Stabilisers and the "Used Together" Rule

E4's first plan was parked after its verification (journal of 2026-10-31, § The E1-E5 Plans, Taken Up Again): its units
took the sign of their input, so no rescaling of a unit's weights could change what it did, and a stabiliser could not
act by construction. STATE.md, *Blocked*, asked for a pilot before the plan is written again: units on which a rescaling
can matter, the stabiliser written as an equation with its output named, a stated rule at rest, and a plan written only
once the pilot shows Hebb's rule alone capturing and a stabiliser able to change it.

This notebook is that pilot. It is a design probe on its own seeds (`[42000, s]` and `[43000, s]`), not a test: the
plan in [E1-E5.md § E4](../docs/hypotheses/experiments/E1-E5.md) runs on fresh seeds. It follows the hunch's own test
([matej-hunches.md § E4](../docs/hypotheses/matej-hunches.md)): co-activate a *pair* many times, then show unrelated
cues - the pair must never capture the activity, and its weight must saturate - and its second clause, that units
resting high must not grow weights without input.

**The network.** 100 rate units, each rate in [0, 1]. A unit's rate settles by 100 damped steps from zero,
r <- 0.5 r + 0.5 clip(W r + u + b, 0, 1), where u is the external input and b the unit's resting drive. With no
weights and no input a unit rests at r = b, so *the rule at zero input* is stated by the model itself: a unit with no
input sits at its resting rate. Rescaling a unit's incoming weights changes its rate, so a stabiliser can act.

**Training.** 300 trials in an order drawn from the seed: 200 *pair* trials, on which the pair (units 0 and 1) gets
u = 1, and 100 *background* trials, on which five of the other units (2 to 79), drawn from the seed, get u = 1. After
each trial settles, the weights change by the rule, using the settled rates; self-weights stay zero.

**The rules**, each an equation with its output named - y_i is unit i's settled rate on the trial, eta = 0.05:

- *Hebb*: dW_ij = eta y_i y_j.
- *Oja* ([Oja 1982](https://doi.org/10.1007/BF00275687)): dW_ij = eta y_i (y_j - y_i W_ij), with y the settled rate, not
  a clamped pattern.
- *Row normalisation*: Hebb, then each unit's incoming weights scaled down so their L2 norm is at most 1.
- *BCM* ([Bienenstock, Cooper and Munro 1982](https://doi.org/10.1523/JNEUROSCI.02-01-00032.1982)):
  dW_ij = eta y_i (y_i - theta_i) y_j, with theta_i a running mean of y_i squared (theta <- 0.9 theta + 0.1 y^2,
  starting at 0.1).
- *Homeostatic scaling* ([Turrigiano 2008](https://doi.org/10.1016/j.cell.2008.10.008)): Hebb, then each unit's
  incoming weights multiplied by max(0, 1 + gamma (target - r_i)), gamma = 0.5, target 0.1, where r_i is the unit's
  settled rate on the trial, whatever the "used together" rule says - a unit firing above its target scales its inputs
  down.

**The "used together" rule**, on any of them: in the update, y_i is replaced by y_i times [u_i > 0], so a pair counts
only when both units were driven by the trial's input - resting activity, however high, adds nothing.

**Conditions.** *Pair*: as above, every unit resting at b = 0.05. *Hub*: no pair trials, and 20 units (80 to 99) rest
high, at b = 0.6, throughout. Their *no-pair control*: the same rule, seed and background trials, without the pair
trials and with every unit resting at 0.05.

**Measures**, on 100 unrelated cues (five of units 2 to 79 driven, drawn from a separate seed): the pair *captures* a
cue when both its units settle above 0.5; the hub captures it when the hub's mean rate settles at or above 0.9 (it rests
at 0.6); the share of units above 0.5; and the weight W_01 over training, which saturates when it stops growing.


In [1]:
import os

import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

N, PAIR, HUB = 100, (0, 1), np.arange(80, 100)
ETA, REST, HUB_REST = 0.05, 0.05, 0.6
GAMMA, TARGET = 0.5, 0.1
PAIR_TRIALS, BG_TRIALS, K, CUES = 200, 100, 5, 100
OTHERS = np.arange(2, 80)


def settle(W, u, rest, steps=100):
    """The settled rates: 100 damped steps from zero."""
    r = np.zeros(N)
    for _ in range(steps):
        r = 0.5 * r + 0.5 * np.clip(W @ r + u + rest, 0.0, 1.0)
    return r


def train(rng, rule, pair_trials, hub_high, used, eta=ETA, rest_level=REST, gamma=GAMMA, target=TARGET):
    """Weights after training, the resting drives, and W_01 after every trial."""
    W = np.zeros((N, N))
    rest = np.full(N, rest_level)
    if hub_high:
        rest[HUB] = HUB_REST
    theta = np.full(N, 0.1)
    trials = ["pair"] * pair_trials + ["bg"] * BG_TRIALS
    rng.shuffle(trials)
    w01 = []
    for kind in trials:
        u = np.zeros(N)
        if kind == "pair":
            u[list(PAIR)] = 1.0
        else:
            u[rng.choice(OTHERS, K, replace=False)] = 1.0
        rate = settle(W, u, rest)
        y = rate * (u > 0) if used else rate
        if rule in ("hebb", "norm", "scale"):
            dW = eta * np.outer(y, y)
        elif rule == "oja":
            dW = eta * (np.outer(y, y) - (y ** 2)[:, None] * W)
        elif rule == "bcm":
            dW = eta * np.outer(y * (y - theta), y)
            theta = 0.9 * theta + 0.1 * y ** 2
        W = W + dW
        if rule == "norm":
            W = W / np.maximum(np.linalg.norm(W, axis=1), 1.0)[:, None]
        if rule == "scale":  # scaled by the rate the unit fired at on this trial, unmasked
            W = W * np.clip(1 + gamma * (target - rate), 0.0, None)[:, None]
        np.fill_diagonal(W, 0.0)
        w01.append(W[PAIR[0], PAIR[1]])
    return W, rest, np.array(w01)


def test(rng, W, rest):
    """Shares of 100 unrelated cues on which the pair captures, the hub captures, and of units above 0.5."""
    pair, hub, active = [], [], []
    for _ in range(CUES):
        u = np.zeros(N)
        u[rng.choice(OTHERS, K, replace=False)] = 1.0
        r = settle(W, u, rest)
        pair.append(r[PAIR[0]] > 0.5 and r[PAIR[1]] > 0.5)
        hub.append(r[HUB].mean() >= 0.9)
        active.append((r > 0.5).mean())
    return float(np.mean(pair)), float(np.mean(hub)), float(np.mean(active))


# Harness checks: with no weights a unit rests at its resting drive, and a driven unit saturates.
assert np.allclose(settle(np.zeros((N, N)), np.zeros(N), np.full(N, REST)), REST)
assert np.allclose(settle(np.zeros((N, N)), np.ones(N), np.full(N, REST)), 1.0)
print("numpy", np.__version__, "- harness checks passed")


numpy 2.5.4 - harness checks passed


## First, the Dead Ends: the First Plan's ±1 Units

Before the rate network, the first plan's own network: 200 units taking the sign of their input (a unit with exactly
zero input keeps its state), ten background patterns and a frequent pattern P trained 50 times, cues with every overlap
below 0.1, capture when a settled state's absolute overlap with P is at least 0.9, and the control trained with P once.
Two stabilisers that act on a weight's size: Oja's update with the unit's output clamped to the presented state, which
makes each weight an exponential average of the patterns' outer products, and clipping every weight at c / N. Seeds
`[41000, s]` for s = 0 to 5, controls ordered by `[41000, s, 1]`.

In [2]:
def pm_train(pats, seq, rule, param):
    n = pats.shape[1]
    W = np.zeros((n, n))
    for k in seq:
        p = pats[k]
        if rule == "hebb":
            W += np.outer(p, p) / n
        elif rule == "ema":
            W = (1 - param) * W + param * np.outer(p, p) / n
        elif rule == "clip":
            W = np.clip(W + np.outer(p, p) / n, -param / n, param / n)
        np.fill_diagonal(W, 0.0)
    return W


def pm_capture(W, pats, cues):
    S = cues.copy()
    for _ in range(50):
        changed = False
        for i in range(S.shape[1]):
            h = S @ W[:, i]
            new = np.where(h > 0, 1.0, np.where(h < 0, -1.0, S[:, i]))
            changed |= bool(np.any(new != S[:, i]))
            S[:, i] = new
        if not changed:
            break
    return float(np.mean(np.abs(S @ pats[-1]) / S.shape[1] >= 0.9))


def pm_case(s):
    rng = np.random.default_rng([41000, s])
    pats = rng.choice([-1.0, 1.0], size=(11, 200))
    seq = list(range(10)) + [10] * 50
    rng.shuffle(seq)
    seq0 = list(range(11))
    np.random.default_rng([41000, s, 1]).shuffle(seq0)
    cues = []
    while len(cues) < 100:
        c = rng.choice([-1.0, 1.0], size=200)
        if np.all(np.abs(pats @ c) / 200 < 0.1):
            cues.append(c)
    return pats, seq, seq0, np.array(cues)


cases = [pm_case(s) for s in range(6)]
print("rule  param   capture by seed                    control by seed")
for rule, param in [("hebb", None), ("ema", 0.01), ("ema", 0.1), ("ema", 0.3), ("clip", 1), ("clip", 2), ("clip", 4)]:
    cap = [pm_capture(pm_train(p, q, rule, param), p, c) for p, q, q0, c in cases]
    ctl = [pm_capture(pm_train(p, q0, rule, param), p, c) for p, q, q0, c in cases]
    print(f"{rule:5s} {str(param):5s}  {np.round(cap, 2)}  {np.round(ctl, 2)}")


rule  param   capture by seed                    control by seed


hebb  None   [1. 1. 1. 1. 1. 1.]  [0.08 0.06 0.03 0.05 0.03 0.16]


ema   0.01   [1. 1. 1. 1. 1. 1.]  [0.03 0.03 0.08 0.12 0.03 0.22]


ema   0.1    [1. 1. 1. 1. 1. 1.]  [0.   0.   0.19 0.54 0.1  0.15]


ema   0.3    [1.   0.99 0.96 0.87 1.   1.  ]  [0.   0.   0.17 0.74 0.03 0.06]
clip  1      [1.   1.   1.   0.58 1.   1.  ]  [0.   0.   0.12 0.89 0.   0.  ]


clip  2      [1.   1.   1.   0.96 1.   1.  ]  [0.   0.   0.23 0.73 0.08 0.06]


clip  4      [1. 1. 1. 1. 1. 1.]  [0.   0.01 0.16 0.53 0.04 0.11]


## The Pilot Table

Every rule, with and without the "used together" rule, in both conditions, on pilot seeds `[42000, s]` for s = 0 to 4,
cues from `[42000, s, 9]`. For each cell: the shares of cues the pair captures (and its control), the hub captures, the
mean share of units above 0.5, and W_01 after training - the mean and range over the five seeds.

In [3]:
RULES = ["hebb", "oja", "norm", "bcm", "scale"]
SEEDS = range(5)
rows = {}
print(f"{'rule':6s} {'cond':5s} {'used':5s}  pair  (ctrl)   hub  (ctrl)  active   W_01 mean [min, max]")
for rule in RULES:
    for cond in ("pair", "hub"):
        for used in (False, True):
            got = []
            for s in SEEDS:
                W, rest, w01 = train(np.random.default_rng([42000, s]), rule, PAIR_TRIALS if cond == "pair" else 0, cond == "hub", used)
                Wc, rc, _ = train(np.random.default_rng([42000, s]), rule, 0, False, used)
                p, h, a = test(np.random.default_rng([42000, s, 9]), W, rest)
                pc, hc, _ = test(np.random.default_rng([42000, s, 9]), Wc, rc)
                got.append((p, pc, h, hc, a, w01[-1]))
            got = np.array(got)
            rows[(rule, cond, used)] = got
            m = got.mean(axis=0)
            print(f"{rule:6s} {cond:5s} {str(used):5s}  {m[0]:.2f}  ({m[1]:.2f})  {m[2]:.2f}  ({m[3]:.2f})  {m[4]:.2f}   "
                  f"{m[5]:.3f} [{got[:, 5].min():.3f}, {got[:, 5].max():.3f}]")


rule   cond  used   pair  (ctrl)   hub  (ctrl)  active   W_01 mean [min, max]


hebb   pair  False  1.00  (1.00)  1.00  (1.00)  1.00   14.893 [14.844, 14.953]


hebb   pair  True   1.00  (0.00)  0.00  (0.00)  0.76   10.000 [10.000, 10.000]


hebb   hub   False  1.00  (1.00)  1.00  (1.00)  1.00   4.837 [4.834, 4.838]


hebb   hub   True   0.00  (0.00)  0.00  (0.00)  0.93   0.000 [0.000, 0.000]


oja    pair  False  1.00  (1.00)  1.00  (1.00)  1.00   1.000 [1.000, 1.000]


... (11 outputs left out)

scale  pair  False  1.00  (1.00)  1.00  (1.00)  1.00   0.061 [0.061, 0.061]


scale  pair  True   0.00  (0.00)  0.00  (0.00)  0.05   0.063 [0.061, 0.065]


scale  hub   False  1.00  (1.00)  1.00  (1.00)  1.00   0.061 [0.061, 0.061]


scale  hub   True   0.00  (0.00)  0.00  (0.00)  0.25   0.000 [0.000, 0.000]


## How Far the Pattern Holds

The table rests on one setting of the constants. The sweep varies them - eta in {0.02, 0.05, 0.1}, the resting drive in
{0.02, 0.05, 0.1}, 50 or 200 pair trials, and scaling's gamma and target over five pairs - for the two variants the
table separates most sharply: Hebb with the "used together" rule, and homeostatic scaling with it. Each entry is the
pair's capture excess over its control, the mean over pilot seeds `[43000, s]`, s = 0 to 2.

In [4]:
import itertools

sweep = []
for gamma, target in [(0.2, 0.1), (0.5, 0.1), (1.0, 0.1), (0.5, 0.05), (0.5, 0.2)]:
    for eta, rest_level, pair_trials in itertools.product([0.02, 0.05, 0.1], [0.02, 0.05, 0.1], [50, 200]):
        excess = {}
        for rule in ("hebb", "scale"):
            e = []
            for s in range(3):
                kw = dict(eta=eta, rest_level=rest_level, gamma=gamma, target=target)
                W, rest, _ = train(np.random.default_rng([43000, s]), rule, pair_trials, False, True, **kw)
                Wc, rc, _ = train(np.random.default_rng([43000, s]), rule, 0, False, True, **kw)
                e.append(test(np.random.default_rng([43000, s, 9]), W, rest)[0] - test(np.random.default_rng([43000, s, 9]), Wc, rc)[0])
            excess[rule] = float(np.mean(e))
        sweep.append((gamma, target, eta, rest_level, pair_trials, excess["hebb"], excess["scale"]))
sweep = np.array(sweep)
print(f"{len(sweep)} settings")
print("Hebb + used: excess >= 0.95 in", int((sweep[:, 5] >= 0.95).sum()), "; <= 0.05 in", int((sweep[:, 5] <= 0.05).sum()))
print("scaling + used: excess >= 0.95 in", int((sweep[:, 6] >= 0.95).sum()), "; <= 0.05 in", int((sweep[:, 6] <= 0.05).sum()))
for row in sweep:
    if row[6] > 0.05 or row[5] < 0.95:
        print("  exception:", dict(zip(["gamma", "target", "eta", "rest", "pair_trials", "hebb", "scale"], np.round(row, 3))))


90 settings
Hebb + used: excess >= 0.95 in 90 ; <= 0.05 in 0
scaling + used: excess >= 0.95 in 0 ; <= 0.05 in 90


## The Figure

`docs/figures/e4_pilot_w01.svg`: W_01, the pair's weight, after every training trial, for each rule with the "used
together" rule, pilot seed `[42000, 0]`. Drawn by matplotlib with a fixed `svg.hashsalt`, text kept as text and no
date; the cell draws it twice and compares, and reports the file *new*, *changed* or *unchanged*.

In [5]:
import hashlib
import io

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["svg.hashsalt"] = "e4"
plt.rcParams["svg.fonttype"] = "none"
FIGURE = "docs/figures/e4_pilot_w01.svg"
STYLE = {"hebb": "-", "oja": "--", "norm": "-.", "bcm": ":", "scale": (0, (5, 1, 1, 1, 1, 1))}
NAMES = {"hebb": "Hebb", "oja": "Oja", "norm": "row normalisation", "bcm": "BCM", "scale": "homeostatic scaling"}
trajectories = {rule: train(np.random.default_rng([42000, 0]), rule, PAIR_TRIALS, False, True)[2] for rule in RULES}


def draw():
    fig, ax = plt.subplots(figsize=(7.0, 3.6))
    for rule in RULES:
        ax.plot(np.arange(1, len(trajectories[rule]) + 1), trajectories[rule], linestyle=STYLE[rule], color="#1d1d1d",
                linewidth=1.4, label=NAMES[rule])
    ax.set_yscale("symlog", linthresh=0.1)
    ax.set_xlabel("training trial")
    ax.set_ylabel("W_01, the pair's weight")
    ax.set_title("E4's pilot: the pair's weight under each rule, with the \"used together\" rule", fontsize=10)
    ax.grid(True, color="#e4e4e4", linewidth=0.6)
    ax.legend(fontsize=8, frameon=False)
    fig.tight_layout()
    out = io.StringIO()
    fig.savefig(out, format="svg", metadata={"Date": None})
    plt.close(fig)
    return out.getvalue()


first, second = draw(), draw()
assert first == second, "the figure differs between two drawings in one run"
before = open(FIGURE, encoding="utf-8").read() if os.path.exists(FIGURE) else None
status = "new" if before is None else ("unchanged" if before == first else "changed")
with open(FIGURE, "w", encoding="utf-8", newline="\n") as f:
    f.write(first)
print(f"{FIGURE}: {status}, {len(first.encode('utf-8'))} bytes, sha256 {hashlib.sha256(first.encode('utf-8')).hexdigest()}")


docs/figures/e4_pilot_w01.svg: unchanged, 31081 bytes, sha256 4fc5f073750e27882df188f0b145d81826ffc26ece759e66ec8304a00df4b1b8


## What the Pilot Shows

- **Hebb's rule alone captures.** With the "used together" rule, so that resting activity adds nothing, the trained
  pair captures every unrelated cue while its control captures none, on every pilot seed and in all 90 settings of the
  sweep. Without the rule, every variant - controls included - runs away from resting activity alone.
- **A stabiliser can change it - but only one that bounds activity.** Homeostatic scaling with the "used together" rule
  stops the capture on every seed and in every setting of the sweep, and the network rests near its resting rate.
  Oja's rule, row normalisation and BCM each bound the pair's weight - it saturates, at about 1, 1 and 3.6 - and none
  stops the capture: a pair joined by a saturated weight still switches itself on. The hunch names all three as
  stabilisers; on this network they bound the weight, which its test asks for, and not the capture, which it also asks
  for.
- **The "used together" rule stops the hub.** With it, the hub's resting activity grows no weights and the hub never
  captures, under every rule; without it, the hub, and the network, run away.
- **One thing the pilot did not ask about.** With the "used together" rule but no scaling, the background units, driven
  five at a time, wire into a runaway assembly of their own - 76% to 93% of units above 0.5 on an unrelated cue, against
  5% at rest under scaling.

So the plan can be written again: Hebb's rule alone captures and a stabiliser can change it. Its test is close to
foregone at these constants, which the plan says.